# 09 — Robustness

Lookback sensitivity of GMV, MDP, ERC and HRP (H3), the calibration of the 252-day ex-ante vol forecast for the 8 core runs (H4) and the stability of their Sharpe ranks from train to test (H5), under the design pre-registered in §1. 60/40 appears only in §6b, as an unregistered descriptive row.

## Setup

Prices, log returns for estimation, simple returns for the backtest, BIL as $r_f$, the panel the classes read and the 208 rebalance labels.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import maplab as ml
import _nb07_helpers as h7
import _nb08_helpers as h8
import _nb09_helpers as h
from maplab import Panel, apply_style

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

F4 = lambda v: f"{v:.4f}"
E3 = lambda v: f"{v:.3e}"

print(f"prices: {prices.shape[0]:,} days x {prices.shape[1]} assets "
      f"({prices.index.min().date()} -> {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}")

rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"
print(f"rebalance labels: n={len(rdates)}  first={rdates[0].date()}  last={rdates[-1].date()}")

In [ ]:
reg = ml.inference.load_registration("nb09")
ml.inference.render_registration(reg)

## §2 The statistics

General forms of what §1 adds to notebook 08. $x_t$ is a run's daily net return in excess of BIL. The Newey–West covariance (Newey & West 1987, 1994), the delta-method Sharpe difference (Jobson & Korkie 1981; Memmel 2003; HAC form as in Ledoit & Wolf 2008) and the Holm adjustment (Holm 1979) are as defined in notebook 08 §2.

**Lookback ΔSR on a common start.** For method $m$ and lookback $L \in \lbrace 504, 756 \rbrace$,
$$\hat\Delta_{m,L} = \widehat{SR}\big(x^{(m,L)}\big) - \widehat{SR}\big(x^{(m,252)}\big),$$
with both runs scored from 2011-01-31, so the two series cover the same days. Its standard error is the notebook 08 delta-method form with $y_t = \big(x^{(m,L)}_t,\ x^{(m,252)}_t,\ (x^{(m,L)}_t)^2,\ (x^{(m,252)}_t)^2\big)$. A method's two tests share its 252-day run, so the 8 statistics are dependent; Holm's adjustment holds under any dependence.

**Variance ratio and Jensen gap.** Holding segment $k = 1,\dots,K$ has realized vol $s_k$ and ex-ante vol $\hat\sigma_k = \sqrt{w_k'\hat\Sigma_k w_k}$ (§1). With $v_k = s_k^2/\hat\sigma_k^2$ and $q_k = \log(s_k/\hat\sigma_k) = \tfrac12\log v_k$,
$$\bar v = \frac1K\sum_{k=1}^K v_k, \qquad \bar q = \tfrac12\log\bar v - J, \qquad J = \tfrac12\Big(\log\bar v - \frac1K\sum_{k=1}^K\log v_k\Big) \ge 0 .$$
$J \ge 0$ is Jensen's inequality for the concave log (Jensen 1906), with equality only when all $v_k$ are equal. If a segment's daily returns are serially uncorrelated and their variance equals the forecast, $E[s_k^2] = \hat\sigma_k^2$, so $E[\bar v] = 1$ while $E[\bar q] < 0$: a negative $\bar q$ alone does not show an over-forecast, and $\bar v$ carries no such bias.

**Segment block bootstrap** (Künsch 1989; circular blocks as in Politis & Romano 1992). The $v_k$ in time order are resampled in blocks of $b = 12$ consecutive segments: $\lceil K/b\rceil$ block starts are drawn uniformly from $1,\dots,K$, each block wraps from segment $K$ back to segment 1, and the joined blocks are cut to length $K$. The blocks keep the dependence between neighbouring segments. From $B$ resampled means $\bar v^*$: $t = (\bar v - 1)/\mathrm{sd}(\bar v^*)$ and the 95% percentile interval.

**Spearman rank correlation** (Spearman 1904). With $R_i$ and $S_i$ the ranks of run $i$'s train and test Sharpe ratios, $\rho_S$ is the Pearson correlation of the ranks; without ties $\rho_S = 1 - 6\sum_i (R_i - S_i)^2/\big(n(n^2-1)\big)$. With $n = 8$ runs it is descriptive.

## §3 Core runs and reproduction gate

The 8 notebook 08 core runs at 252 days from the default start (208 rebalances, 40 in test), gated on their Phase 1 Sharpe ratios in the three windows. They feed H4 and H5.

In [ ]:
CORE = h.CORE
core = h.build_core_runs(simple_returns, panel)
for name, r in core.items():
    assert r["wlog"].index.equals(rdates), f"{name}: wlog index != rdates"
print(f"{len(core)} core runs completed: {', '.join(core)}")

core_summary = h8.summary_table(core, CORE, split_ts, rf_daily)
ml.inference.check_reproduction(core_summary, {m: h8.REPRO_KEYS[m] for m in CORE})
print("GATES PASS")

## §4 Lookback grid

GMV, MDP, ERC and HRP (sample Σ) at 252, 504 and 756 days, all scored from the common start 2011-01-31 (184 rebalances, 40 in test), with a weight cross-check against §3 and the summary table in the three windows.

In [ ]:
common_start = h.common_start(log_returns.index, 756)
cs_dates = ml.rebalance_dates(simple_returns.index)
cs_dates = cs_dates[cs_dates >= common_start]
print(f"common start: {common_start.date()}  rebalances: {len(cs_dates)}  in test: {int((cs_dates > split_ts).sum())}")
assert common_start == pd.Timestamp("2011-01-31")
assert len(cs_dates) == 184 and int((cs_dates > split_ts).sum()) == 40

GRID = h.GRID
grid = h.build_grid_runs(simple_returns, panel, common_start)
for name, r in grid.items():
    assert r["wlog"].index.equals(cs_dates), f"{name}: wlog index != common-start rebalances"
    assert r["diag"]["n_rebalances"] == 184
print(f"{len(grid)} grid runs completed: {', '.join(grid)}")

Grid 252-day target weights against the §3 target weights on the 184 common rebalance dates.

In [ ]:
dw = h.weight_crosscheck(grid, core)
for m, v in dw.items():
    print(f"{m}: max |dw| (grid 252d vs default start) = {E3(v)}")

The 252-day grid weights are identical to the §3 weights on all 184 common dates, so the grid's 252-day runs differ from the core runs only in their later start.

Summary table of the 12 grid runs in the three windows of the common-start sample; five of its columns are shown.

In [ ]:
grid_summary = h8.summary_table(grid, GRID, split_ts, rf_daily)

Xg = h8.excess_frame(grid, GRID, rf_daily)
XgW = h8.window_frames(Xg, split_ts)
Xg_full, Xg_test = XgW["full"], XgW["test (> split)"]
seg0 = simple_returns.index[simple_returns.index >= common_start][0]
assert Xg_full.index[0] == seg0
for w, Xw in XgW.items():
    print(f"{w:<18} T={len(Xw):>5}  first={Xw.index[0].date()}  last={Xw.index[-1].date()}  "
          f"NW lag={ml.robust.nw_lag(len(Xw))}")
print()
print(grid_summary[["ann_return", "ann_vol", "sharpe", "max_dd", "ann_turnover"]].to_string(float_format=F4))

In [ ]:
start_cmp = pd.DataFrame({
    "grid 252d (common start)": [grid_summary.loc[(h.grid_label(m, 252), "full"), "sharpe"] for m in h.GRID_METHODS],
    "Phase 1 (default start)": [core_summary.loc[(h.P1_LABEL[m], "full"), "sharpe"] for m in h.GRID_METHODS],
}, index=h.GRID_METHODS)
start_cmp["diff"] = start_cmp["grid 252d (common start)"] - start_cmp["Phase 1 (default start)"]
print("full-window Sharpe, 252-day runs:")
print(start_cmp.to_string(float_format=F4))

Starting two years later lowers each method's 252-day full-window Sharpe by 0.03 (MDP) to 0.09 (ERC) against §3, the same order as the lookback differences tested in §5 (+0.06 to +0.10).

## §5 H3 — lookback (registered)

Delta-method Newey–West test of each method's 504-day and 756-day Sharpe ratio against its 252-day Sharpe ratio on the common-start full window, Holm-adjusted over the 8 tests, with the test-window difference.

In [ ]:
T_full, T_test = len(Xg_full), len(Xg_test)
rows = []
for m in h.GRID_METHODS:
    base = h.grid_label(m, 252)
    for L in (504, 756):
        lab = h.grid_label(m, L)
        r = ml.robust.sharpe_diff_hac(Xg_full[lab], Xg_full[base])
        assert r["L"] == ml.robust.nw_lag(T_full)
        rt = ml.robust.sharpe_diff_hac(Xg_test[lab], Xg_test[base])
        assert rt["L"] == ml.robust.nw_lag(T_test)
        rows.append({"id": f"H3_{m}_{L}", "method": m, "lookback": L, "dSR_full": r["d_sr"], "se": r["se"],
                     "t": r["t"], "p_raw": r["p"], "dSR_test": rt["d_sr"],
                     "same_sign_test": bool(np.sign(rt["d_sr"]) == np.sign(r["d_sr"]))})

h3 = pd.DataFrame(rows).set_index("id")
h3["p_holm"] = ml.robust.holm(h3["p_raw"].to_numpy())
h3_table = h3[["dSR_full", "se", "t", "p_raw", "p_holm", "dSR_test", "same_sign_test"]]
print(h3_table.to_string(float_format=F4))

h3_stats = {}
for hid, row in h3.iterrows():
    h3_stats[hid] = dict(
        display=f"ΔSR {row['dSR_full']:+.2f}, t {row['t']:.2f}, Holm p {row['p_holm']:.3f}",
        p_adj=float(row["p_holm"]), same_sign_test=bool(row["same_sign_test"]),
    )
reg_ids = {x["id"] for x in reg["hypothesis"]}
assert len(h3_stats) == 8 and set(h3_stats) <= reg_ids
assert set(h3_stats) == {x["id"] for x in reg["hypothesis"] if x["id"].startswith("H3_") and x["kind"] == "adjusted_null"}

All 8 lookback differences are positive (+0.06 to +0.10), and all 8 keep their sign in the test window. The largest t is 2.03 (ERC 504d); after Holm over the 8 tests its p is 0.338, the smallest of the 8. Each method's 504-day and 756-day tests share its 252-day run, so the 8 statistics are dependent.

H3 descriptive: annual turnover, median effN, median UUP target weight and UUP target weight at 2022-12-31 per method and lookback.

In [ ]:
h3_desc = h.h3_desc_table(grid, grid_summary, "2022-12-31")
print(h3_desc.to_string(float_format=F4))

turn_cmp = pd.DataFrame({
    "grid 252d (common start)": [grid_summary.loc[(h.grid_label(m, 252), "full"), "ann_turnover"] for m in h.GRID_METHODS],
    "Phase 1 (default start)": [core_summary.loc[(h.P1_LABEL[m], "full"), "ann_turnover"] for m in h.GRID_METHODS],
}, index=h.GRID_METHODS)
print()
print("ann_turnover, full window:")
print(turn_cmp.to_string(float_format=F4))

From 252 to 756 days annual turnover goes GMV 71% → 29%, MDP 50% → 24%, ERC 26% → 15%, HRP 62% → 28%; the 756-day level is 41–57% of the 252-day level for all 4 methods. The UUP weight at 2022-12-31 falls for all 4 methods (GMV 58.1% → 46.6%, MDP 59.2% → 49.4%, ERC 48.4% → 38.5%, HRP 55.3% → 50.7%), while each method's median UUP weight over the 184 rebalances moves by at most 2.3 pp across lookbacks. The common start changes 252-day turnover by at most 2 pp against §3 (second table).

### Figure F6 — full-window Sharpe, annual turnover and median effN by lookback, one line per method.

In [ ]:
sharpe_full = grid_summary.xs("full", level="window")["sharpe"]
fig = h.lookback_panels(h3_desc, sharpe_full)
plt.show()

Turnover falls with each longer lookback for all 4 methods; median effN moves by at most 0.36 from its 252-day value (ERC at 756 days).

### Figure F7 — UUP target weight at each rebalance by lookback, one panel per method; the dotted line marks the train/test split.

In [ ]:
fig = h.uup_paths(grid, split_ts)
plt.show()

rows = []
for lab in GRID:
    wt = grid[lab]["wlog"].loc[grid[lab]["wlog"].index > split_ts]
    med = wt.median()
    rows.append({"run": lab, "n_test": len(wt), "median_w_UUP_test": float(med["UUP"]),
                 "largest_median": med.idxmax()})
uup_test = pd.DataFrame(rows).set_index("run")
print("test window (> split): median UUP target weight and the asset with the largest median weight")
print(uup_test.to_string(float_format=F4))

Over the 40 test rebalances UUP has the largest median weight in all 12 grid runs. Its test-window median weight is 45.1–54.0% across the 4 methods at 756 days and 44.4–52.8% at 252 days.

## §6 H4 — forecast bias (registered)

Realized over ex-ante volatility per holding segment for the 8 core runs, with the notebook 07 segment construction and a reproduction gate against its printed means of q.

In [ ]:
q_table, n_dropped, _ = h7.forecast_bias_table(panel, rdates, core, simple_returns, CORE)
print(f"segments dropped (< 10 remaining days after excluding cost-booking day): {n_dropped}")
n_seg = q_table.groupby("method").size().reindex(CORE)
assert n_seg.nunique() == 1, n_seg.to_dict()
print(f"segments per method: {int(n_seg.iloc[0])}")

Q_PHASE1 = {"HRP(S)": -0.0999, "GMV(S)": -0.0356, "MDP(S)": -0.0518, "ERC(S)": -0.1049, "EW": -0.1885}
q_mean = q_table.groupby("method")["q"].mean()
for m, exp in Q_PHASE1.items():
    got = round(float(q_mean[m]), 4)
    assert got == exp, f"H4 GATE FAILED: {m} mean q {got} != {exp}"
print("H4 GATE PASS")

Mean variance ratio per method with its segment block-bootstrap standard error and interval, on the full and test windows.

In [ ]:
B4, BLOCK4, SEED4 = 10000, 12, 20260923
h4_rows, dec_rows = [], []
for m in CORE:
    seg = q_table[q_table["method"] == m].sort_values("asof")
    test = seg[seg["asof"] > split_ts]
    d = ml.robust.forecast_bias_decomposition(seg["realized_vol"], seg["exante_vol"])
    dt = ml.robust.forecast_bias_decomposition(test["realized_vol"], test["exante_vol"])
    assert abs(d["q_bar"] - float(q_mean[m])) <= 1e-12
    v_full = ((seg["realized_vol"] / seg["exante_vol"]) ** 2).to_numpy()
    draws = ml.robust.segment_block_bootstrap_mean(v_full, block=BLOCK4, B=B4, seed=SEED4)
    sd = float(draws.std(ddof=1))
    lo, hi = np.percentile(draws, [2.5, 97.5])
    h4_rows.append({"method": m, "vbar_full": d["vbar"], "sd": sd, "t": (d["vbar"] - 1) / sd,
                    "lo": float(lo), "hi": float(hi), "vbar_test": dt["vbar"],
                    "n_full": len(seg), "n_test": len(test)})
    for w, dd in (("full", d), ("test", dt)):
        dec_rows.append({"method": m, "window": w, "q_bar": dd["q_bar"],
                         "half_log_vbar": dd["half_log_vbar"], "J": dd["J"]})

h4 = pd.DataFrame(h4_rows).set_index("method")
print(h4.to_string(float_format=F4))

### Figure F8a — full-window mean variance ratio with the segment block-bootstrap 95% interval; the dashed line marks 1.

In [ ]:
f8a = h4.reset_index()[["method", "vbar_full", "lo", "hi"]]
fig = ml.plotting.forest_plot(f8a, "method", "vbar_full", "lo", "hi", ref=1.0,
                              xlabel="mean realized/ex-ante variance ratio (full window)")
plt.show()

v̄ is above 1 for all 8 methods, from 1.30 (EW) to 1.73 (MaxSharpe): on average the ex-ante vol under-forecasts realized variance, the opposite of the registered claim. Only MaxSharpe (t +2.34) reaches an absolute t of 2, with the sign opposite to the claim. The test-window v̄ is below 1 only for EW (0.90). The 95% interval lies above 1 for GMV, MaxSharpe and MDP; contains 1 for the other 5. For GMV and MDP the percentile intervals exclude 1 while t < 2: the bootstrap distribution of v̄ has a shorter lower tail than a normal one.

H4 descriptive: mean log ratio, half log mean variance ratio and Jensen gap per method, full and test windows.

In [ ]:
h4_dec = pd.DataFrame(dec_rows).set_index(["method", "window"])
print(h4_dec.to_string(float_format=F4))

H4_KEYS = {"EW": "EW", "GMV(S)": "GMV", "MaxSharpe(S)": "MaxSharpe", "BMV(0.3)": "BMV",
           "BL(0.1,S)": "BL", "MDP(S)": "MDP", "ERC(S)": "ERC", "HRP(S)": "HRP"}
assert list(H4_KEYS) == CORE
h4_stats = {}
for m, key in H4_KEYS.items():
    v, t, vt = float(h4.loc[m, "vbar_full"]), float(h4.loc[m, "t"]), float(h4.loc[m, "vbar_test"])
    h4_stats[f"H4_{key}"] = dict(display=f"vbar {v:.3f}, t {t:.2f}, test vbar {vt:.3f}",
                                 t_full=t, mean_test=vt - 1)
assert set(h4_stats) <= reg_ids
assert set(h4_stats) == {x["id"] for x in reg["hypothesis"] if x["id"].startswith("H4_") and x["kind"] == "directional_t"}

### Figure F8b — full-window decomposition of the mean log ratio into half log mean variance ratio and Jensen gap.

In [ ]:
fig = h.bias_decomposition_bars(h4_dec.xs("full", level="window").loc[CORE])
plt.show()

½·log v̄ is positive for all 8 methods and q̄ is negative for 7 of the 8 methods (not MaxSharpe), so every negative q̄ comes from J: J runs from 0.20 to 0.37, ½·log v̄ from 0.13 to 0.27. For notebook 07's H7, the HRP − GMV difference in q̄ of −0.064 splits into +0.010 from ½·log v̄ and −0.074 from J; in variance terms the two are close (v̄ 1.53 for HRP, 1.50 for GMV; not tested).

### §6b Variance-ratio profile (descriptive, not registered — added after the H4 results)

The typical segment and the largest segments of v = (realized / ex-ante)² per core method, and v̄ without the segment starting 2020-02-29.

In [ ]:
vr = h.variance_ratio_profile(q_table, CORE)
print(vr.to_string(float_format=lambda v: f"{v:.4f}"))

The 252-day forecast is too high in the typical segment and far too low in a few. Median v is below 1 for all 8 methods (0.62 to 0.97), and 53–70% of segments have v < 1. The segment starting 2020-02-29 is the largest for all 8 methods (v from 37 for MDP to 74 for HRP); the segment starting 2025-03-31 is second for 7 of the 8 (otherwise: MaxSharpe 2018-01-31). The five largest segments carry 23–38% of the sum of v, and without the 2020 segment v̄ is 0.98 to 1.42, below 1 for EW.

**60/40 (exploratory, unregistered).** The same segments and profile for 60/40 (SPY/IEF, as in notebooks 01–08), after its Sharpe gate. It enters no test and no verdict.

In [ ]:
bench = h8.build_benchmarks(simple_returns, panel)
BENCH = h8.BENCH
for name, r in bench.items():
    assert r["wlog"].index.equals(rdates), f"{name}: wlog index != rdates"
bench_summary = h8.summary_table(bench, BENCH, split_ts, rf_daily)
ml.inference.check_reproduction(bench_summary, h8.BENCH_REPRO_KEYS)
print("60/40 GATE PASS")

q_bench, n_dropped_b, _ = h7.forecast_bias_table(panel, rdates, bench, simple_returns, BENCH)
assert n_dropped_b * len(CORE) == n_dropped, (n_dropped_b, n_dropped)
assert len(q_bench) == int(n_seg.iloc[0])
assert q_bench["asof"].tolist() == q_table.loc[q_table["method"] == CORE[0], "asof"].tolist()
d_b = ml.robust.forecast_bias_decomposition(q_bench["realized_vol"], q_bench["exante_vol"])
vr_b = h.variance_ratio_profile(q_bench, BENCH)
vr_b.insert(0, "vbar", d_b["vbar"])
vr_b["q_bar"] = d_b["q_bar"]
vr_b["J"] = d_b["J"]
print(vr_b.to_string(float_format=F4))

For 60/40 v̄ is 1.27, median v is 0.60 and 71% of segments have v < 1; its largest segment starts 2020-02-29 (v 43). The same pattern as the core runs: too high in the typical segment, too low on average.

## §7 H5 — train vs test Sharpe ranks (descriptive)

Train and test Sharpe ratios of the 8 §3 core runs, their ranks and the Spearman rank correlation.

In [ ]:
h5 = pd.DataFrame({
    "SR_train": core_summary.xs("train (<= split)", level="window").loc[CORE, "sharpe"],
    "SR_test": core_summary.xs("test (> split)", level="window").loc[CORE, "sharpe"],
})
h5["rank_train"] = h5["SR_train"].rank(ascending=False)
h5["rank_test"] = h5["SR_test"].rank(ascending=False)
print(h5.to_string(float_format=F4))
rho_s = float(h5["SR_train"].corr(h5["SR_test"], method="spearman"))
print(f"\nSpearman rank correlation, train vs test Sharpe (8 core runs): {rho_s:.4f}")

The Spearman rank correlation is 0.50. The largest rise is MaxSharpe from 6th to 1st and the largest fall BL(0.1) from 5th to 8th. MDP keeps its rank (3rd). With 8 runs it is descriptive; no test is registered.

### Figure F9 — annualized volatility and return of each core run, train point to test point.

In [ ]:
fig = h.train_test_arrows(core_summary, CORE)
plt.show()

f9 = pd.DataFrame({(w, c): core_summary.xs(w, level="window").loc[CORE, c]
                   for w in ("train (<= split)", "test (> split)") for c in ("ann_return", "ann_vol")})
print(f9.to_string(float_format=F4))

From train to test, annualized return rises for all 8 runs and annualized volatility rises for 2 of the 8 runs (MaxSharpe and BL(0.1)).

## Verdict table

The §1 rules applied mechanically to the H3 statistics in §5 and the H4 statistics in §6.

In [ ]:
table = ml.inference.evaluate(reg, {**h3_stats, **h4_stats})
assert len(table) == 16
print(table.to_string(index=False))
print()
print(table["verdict"].value_counts().to_string())

## §8 TAKEAWAY — the 8 lookback nulls hold; H4 is not supported, and v̄ points the other way

- **H3: 8 of 8 nulls hold.** Every longer lookback has the higher full-window Sharpe (ΔSR +0.06 to +0.10), and all 8 keep their sign in the test window; the smallest Holm p is 0.338 (ERC 504d, raw t 2.03).
- **Lookback changes turnover and the 2022-12-31 UUP weight (descriptive).** From 252 to 756 days turnover falls to 41–57% of its 252-day level; the 2022-12-31 UUP weight falls for all 4 methods, each median UUP weight moves by at most 2.3 pp, and UUP keeps the largest test-window median weight in all 12 grid runs.
- **H4: 8 of 8 NOT SUPPORTED, and the evidence runs the other way.** v̄ runs from 1.30 (EW) to 1.73 (MaxSharpe); only MaxSharpe (t +2.34) reaches an absolute t of 2, with the sign opposite to the claim. ½·log v̄ is positive for every method, so each negative q̄ comes from the Jensen gap J (0.20 to 0.37). Notebook 07's H7 difference in q̄ of −0.064 splits into +0.010 from ½·log v̄ and −0.074 from J.
- **The forecast errs both ways (§6b, descriptive).** Median v is below 1 for all 8 methods, the segment starting 2020-02-29 is the largest for all 8 methods, and without it v̄ is 0.98 to 1.42 (below 1 for EW). 60/40, unregistered, shows the same pattern (v̄ 1.27, median v 0.60).
- **H5 (descriptive).** Train and test Sharpe ranks correlate at ρ = 0.50; the largest rise is MaxSharpe from 6th to 1st, the largest fall BL(0.1) from 5th to 8th.

Lookback length changes turnover and the UUP weight at a date but no tested Sharpe difference; the trailing 252-day vol forecast is too high in the typical segment and far too low in stress, the error a Phase 3 volatility overlay has to address.

## References

- Holm, S. (1979). A simple sequentially rejective multiple test procedure. Scandinavian Journal of Statistics 6(2), 65–70.
- Jensen, J. L. W. V. (1906). Sur les fonctions convexes et les inégalités entre les valeurs moyennes. Acta Mathematica 30, 175–193.
- Jobson, J. D. & Korkie, B. M. (1981). Performance hypothesis testing with the Sharpe and Treynor measures. Journal of Finance 36(4), 889–908.
- Künsch, H. R. (1989). The jackknife and the bootstrap for general stationary observations. Annals of Statistics 17(3), 1217–1241.
- Ledoit, O. & Wolf, M. (2008). Robust performance hypothesis testing with the Sharpe ratio. Journal of Empirical Finance 15(5), 850–859.
- Memmel, C. (2003). Performance hypothesis testing with the Sharpe ratio. Finance Letters 1(1), 21–23.
- Newey, W. K. & West, K. D. (1987). A simple, positive semi-definite, heteroskedasticity and autocorrelation consistent covariance matrix. Econometrica 55(3), 703–708.
- Newey, W. K. & West, K. D. (1994). Automatic lag selection in covariance matrix estimation. Review of Economic Studies 61(4), 631–653.
- Politis, D. N. & Romano, J. P. (1992). A circular block-resampling procedure for stationary data. In R. LePage & L. Billard (eds.), Exploring the Limits of Bootstrap, 263–270. Wiley.
- Spearman, C. (1904). The proof and measurement of association between two things. American Journal of Psychology 15(1), 72–101.